# Part A 함수 정의

## Step 0. 환경 설정

In [ ]:
!pip install -q torch_geometric

import os, sys, time, math, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')

DRIVE   = '/content/drive/MyDrive/ACK2026_Wafer'
RAW_PKL = f'{DRIVE}/MIR-WM811K/MIR-WM811K/Python/WM811K_defects.pkl'

# npz 는 /content 에 먼저 쓰고 Drive 로 복사한다.
# Drive FUSE 에 큰 파일을 직접 쓰면 느리고 간헐적으로 실패한다.
LOCAL_NPZ = '/content/wafer_graph.npz'
DRIVE_NPZ = f'{DRIVE}/Train_Model/Training_Data/wafer_graph.npz'
os.makedirs(os.path.dirname(DRIVE_NPZ), exist_ok=True)

assert os.path.exists(RAW_PKL), f"원본 pkl 을 찾을 수 없다: {RAW_PKL}"
print("원본 확인 완료:", RAW_PKL)


## Step 1. Wafer_Preprocessing

In [ ]:
"""WM-811K 웨이퍼 맵 전처리 v2.

v1(`Data_Preprocessing/Wafer Data Preprocessing6.ipynb`) 대비 변경점

1. 노이즈 필터를 '다이 피치' 단위로 재정의하고, 삭제 대신 플래그로 보존
   - v1: eps=0.10 (정규화 좌표) -> 웨이퍼 크기에 따라 실제 반경이 1.3~8.6 다이로 요동.
     Center 60%, Scratch 59%, Loc 51% 의 결함 다이가 삭제되고 있었음.
   - v2: DBSCAN 을 다이(die) 단위 좌표에서 수행하여 eps 가 항상 동일한 물리적 거리.
     노이즈로 판정된 점도 지우지 않고 `is_noise` 특징/채널로 남김.

2. 유효 다이(waferMap > 0) 기준 반경 정규화
   - v1: rmax = max(w, h) / 2 로 x, y 를 동시에 나눔 -> 종횡비가 다른 웨이퍼(전체의 66%)
     에서 원이 타원으로 왜곡. 최외곽 반경이 웨이퍼마다 0.955~1.037 로 흔들림.
   - v2: 중심 = 유효 다이 무게중심, R = 유효 다이의 최대 반경.
     정의상 모든 결함 다이가 r <= 1 이 되어 클리핑 손실이 0.

3. 극좌표 이미지를 3채널 밀도 맵으로 교체
   - ch0 결함 밀도 = (빈 내 결함 다이 수) / (빈 내 유효 다이 수)
   - ch1 유효 다이 마스크 = 그 위치에 웨이퍼가 존재하는가
   - ch2 노이즈 판정 결함 밀도
   - v1 은 1채널 이진 대입(`= 1.0`)이라 Edge-Ring 결함의 19% 가 같은 픽셀에 겹쳐 소실됐고,
     "결함 없음"과 "다이 없음"을 구분할 수 없었음.

5. GCN 입력 강화
   - 반경 기반 그래프 대신 k-NN 그래프(k 고정) -> 평균 차수가 0.38~73.8 로 벌어지던 문제 해소.
   - 노드 특징을 (x, y) 2차원에서 11차원으로 확장.

주의: 전처리는 라벨(failureType)을 일절 사용하지 않는다. 패턴별로 파라미터를 다르게 주면
추론 시점에 재현할 수 없는 라벨 누수가 된다.
"""


import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN

# 8대 불량 패턴 라벨 인코딩. 전처리·데이터셋·학습이 공유하므로 여기 한 곳에만 둔다.
LABEL_MAP = {
    "Center": 0, "Donut": 1, "Edge-Loc": 2, "Edge-Ring": 3,
    "Loc": 4, "Random": 5, "Scratch": 6, "Near-full": 7,
}
REVERSE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# 노드 특징 벡터의 열 인덱스. 증강(회전/반전)이 이 순서에 의존하므로 변경 시 주의.
FEAT_X, FEAT_Y = 0, 1          # 회전/반전 대상
FEAT_R = 2
FEAT_COS, FEAT_SIN = 3, 4      # 회전/반전 대상
FEAT_DENS2 = 5
FEAT_DENS4 = 6
FEAT_KNN_DIST = 7
FEAT_LOCAL_RATIO = 8
FEAT_IS_NOISE = 9
FEAT_LOG_N = 10
NUM_NODE_FEATURES = 11

# 그래프 수준 스칼라 (분류기에 직접 주입)
NUM_GRAPH_SCALARS = 4

POLAR_CHANNELS = 3


class WaferPreprocessorV2:
    """웨이퍼 맵 한 장을 (그래프, 극좌표 이미지, 스칼라) 3종 표현으로 변환한다.

    Parameters
    ----------
    r_bins, theta_bins : 극좌표 이미지 해상도.
    knn_k : 각 결함 노드가 연결할 최근접 이웃 수. 방향성 엣지로 저장하고 학습 시 대칭화한다.
    noise_eps_dies : DBSCAN 반경. **다이 개수 단위**라 웨이퍼 크기와 무관하게 일정하다.
    noise_min_samples : DBSCAN 최소 샘플 수.
    drop_noise : True 면 노이즈 판정 결함을 실제로 삭제한다. 기본 False(플래그만 부여).
        Random 패턴은 산발 노이즈 자체가 클래스 정의이고 Scratch 는 폭 1~2 다이의 선이라
        삭제하면 신호가 사라진다. 삭제 대신 모델이 판단하도록 정보만 넘긴다.
    density_radii : 국소 밀도 특징을 계산할 반경(다이 단위).
    """

    # 파라미터 설정
    def __init__(
        self,
        r_bins: int = 64,            # 극좌표계
        theta_bins: int = 64,        #
        knn_k: int = 8,              # k-NN 그래프
        noise_eps_dies: float = 2.0, # DBSCAN 노이즈 필터링
        noise_min_samples: int = 3,  # DBSCAN 노이즈 필터링
        drop_noise: bool = False,
        density_radii: tuple[float, float] = (2.0, 4.0),
        include_polar: bool = True,
    ):
        self.r_bins = r_bins
        self.theta_bins = theta_bins
        self.knn_k = knn_k
        self.noise_eps_dies = noise_eps_dies
        self.noise_min_samples = noise_min_samples
        self.drop_noise = drop_noise
        self.density_radii = density_radii
        self.include_polar = include_polar

    # ------------------------------------------------------------------
    # 웨이퍼 기하 구조 추정
    # ------------------------------------------------------------------
    @staticmethod
    def wafer_geometry(wafer_map: np.ndarray):
        """유효 다이만으로 웨이퍼 원의 중심과 반경을 추정한다.

        배열 중심 `(w/2, h/2)` 과 `max(w, h)/2` 를 쓰던 v1 과 달리 종횡비, 여백,
        다이 크기에 영향받지 않는다. 반환하는 R 은 유효 다이의 최대 반경이므로
        모든 유효 다이(=결함 다이 포함)가 r <= 1 을 만족한다.
        """
        vy, vx = np.nonzero(wafer_map > 0)  # 정상, 불량 다이 좌표 추출
        if len(vx) == 0:
            return None
        cx, cy = vx.mean(), vy.mean()   # 다이들의 cx(x좌표 평균), cy(y좌표 평균) 계산 -> 웨이퍼의 실질적인 무게 중심 추출
        dv = np.hypot(vx - cx, vy - cy) # 무게 중심부터 가장 먼 곳의 다이까지의 거리 계산
        radius = float(dv.max())        # 가장 긴 거리를 radius로 지정
        if radius <= 0:
            return None
        return vx, vy, cx, cy, radius

    # ------------------------------------------------------------------
    def process_wafer(self, wafer_map: np.ndarray) -> dict | None:
        wafer_map = np.asarray(wafer_map)
        geom = self.wafer_geometry(wafer_map)
        if geom is None:
            return None
        vx, vy, cx, cy, radius = geom

        by, bx = np.nonzero(wafer_map == 2)   # 불량 다이만 추출
        if len(bx) == 0:
            return None

        # 다이 단위 좌표(1 = 다이 1개 간격). 거리 기반 파라미터는 전부 이 좌표계에서 정의한다.
        defect_px = np.column_stack((bx - cx, cy - by)).astype(np.float64)    # 불량 다이
        die_px = np.column_stack((vx - cx, cy - vy)).astype(np.float64)       # 모든 다이

        # [개선 1] 다이 단위 DBSCAN -> 웨이퍼 크기와 무관하게 동일한 물리적 필터
        if len(defect_px) >= self.noise_min_samples:
            labels = DBSCAN(
                eps=self.noise_eps_dies, min_samples=self.noise_min_samples
            ).fit(defect_px).labels_
            is_noise = (labels == -1)   # 랜덤 불량 다이는 -1로 지정
        else:
            is_noise = np.ones(len(defect_px), dtype=bool)

        if self.drop_noise and (~is_noise).sum() >= 3:
            defect_px = defect_px[~is_noise]
            is_noise = is_noise[~is_noise]

        node_feat = self._node_features(defect_px, die_px, radius, is_noise)
        edge_index = self._knn_edges(defect_px)
        scalars = np.array(
            [
                np.log1p(len(defect_px)) / 10.0,
                len(defect_px) / len(die_px),
                np.log1p(len(die_px)) / 10.0,
                radius / 100.0,
            ],
            dtype=np.float32,
        )

        result = {
            "node_feat": node_feat.astype(np.float16),
            "edge_index": edge_index.astype(np.int16),  # 노드 수 < 32767 보장
            "scalars": scalars,
            "n_defect": int(len(defect_px)),
            "n_die": int(len(die_px)),
            "radius": float(radius),
            "noise_ratio": float(is_noise.mean()),
        }
        if self.include_polar:
            result["polar"] = self._polar_density(
                defect_px, die_px, radius, is_noise).astype(np.float16)
        return result

    # ------------------------------------------------------------------
    # 노드 특징 11차원
    # ------------------------------------------------------------------
    def _node_features(self, defect_px, die_px, radius, is_noise) -> np.ndarray:
        n = len(defect_px)

        # radius로 각 다이의 x,y 좌표를 나누어 -1 ~ 1 사이로 정규화
        x = defect_px[:, 0] / radius
        y = defect_px[:, 1] / radius
        r = np.hypot(x, y)        # 극좌표계 radius
        theta = np.arctan2(y, x)  # 극좌표계 theta

        defect_tree = cKDTree(defect_px)    # 불량 다이의 트리 구조 생성
        die_tree = cKDTree(die_px)          # 전체 다이의 트리 구조 생성

        r_small, r_large = self.density_radii
        # 자기 자신이 항상 포함되므로 -1
        dens_small = np.array(defect_tree.query_ball_point(defect_px, r_small, return_length=True)) - 1.0
        dens_large = np.array(defect_tree.query_ball_point(defect_px, r_large, return_length=True)) - 1.0
        die_large = np.array(die_tree.query_ball_point(defect_px, r_large, return_length=True))

        # k-NN 평균 거리(다이 단위). 노드가 k+1개 미만이면 있는 만큼만 사용한다.
        k = min(self.knn_k + 1, n)
        if k >= 2:
            dists, _ = defect_tree.query(defect_px, k=k)
            knn_mean = dists[:, 1:].mean(axis=1)
        else:
            knn_mean = np.full(n, 10.0)

        feat = np.zeros((n, NUM_NODE_FEATURES), dtype=np.float32)
        feat[:, FEAT_X] = x
        feat[:, FEAT_Y] = y
        feat[:, FEAT_R] = r
        feat[:, FEAT_COS] = np.cos(theta)
        feat[:, FEAT_SIN] = np.sin(theta)
        feat[:, FEAT_DENS2] = np.log1p(np.clip(dens_small, 0, None))      # 좁은 반경 불량 밀도
        feat[:, FEAT_DENS4] = np.log1p(np.clip(dens_large, 0, None))      # 넓은 반경 불량 밀도
        feat[:, FEAT_KNN_DIST] = np.clip(knn_mean, 0, 20.0) / 20.0        # k개의 이웃 불량까지의 거리 평균
        feat[:, FEAT_LOCAL_RATIO] = np.clip(dens_large, 0, None) / np.maximum(die_large, 1.0)   # 전체 정상 다이 대비 불량 다이의 비율
        feat[:, FEAT_IS_NOISE] = is_noise.astype(np.float32)
        feat[:, FEAT_LOG_N] = np.log1p(n) / 10.0                          # 전체 불량 개수
        return feat

    # ------------------------------------------------------------------
    # k-NN 그래프 생성
    # ------------------------------------------------------------------
    def _knn_edges(self, defect_px) -> np.ndarray:
        n = len(defect_px)
        k = min(self.knn_k + 1, n)
        if k < 2:
            return np.zeros((2, 0), dtype=np.int64)
        _, idx = cKDTree(defect_px).query(defect_px, k=k)
        src = np.repeat(np.arange(n), k - 1)
        dst = idx[:, 1:].reshape(-1)
        return np.vstack((src, dst))

    # ------------------------------------------------------------------
    # 3채널 극좌표 밀도 맵 (CNN 용 극좌표 변환)
    # ------------------------------------------------------------------
    def _bin_indices(self, pts, radius):
        r = np.hypot(pts[:, 0], pts[:, 1]) / radius
        theta = np.arctan2(pts[:, 1], pts[:, 0])
        ri = np.clip((r * self.r_bins).astype(np.int64), 0, self.r_bins - 1)
        # v1 은 (theta_bins - 1) 을 곱해 마지막 빈이 theta == +pi 한 점만 받았다.
        # theta_bins 를 곱하고 modulo 를 취해야 각 빈의 각도 폭이 균일해진다.
        u = (theta + np.pi) / (2 * np.pi) * self.theta_bins
        # +0.5 후 내림 = 반올림. 빈 k 의 '중심'이 theta = -pi + k*2pi/T 에 놓인다.
        # 내림만 쓰면 빈 경계가 theta = 0, +-pi/2, +-pi 위에 정확히 얹히는데,
        # 격자 좌표의 결함은 이 축 위에 대량으로 존재해서 좌우 반전 시
        # 그래프 트랙과 이미지 트랙이 반 칸 어긋난다(축 위 점은 반전의 고정점이어야 함).
        ti = np.floor(u + 0.5).astype(np.int64) % self.theta_bins
        return ri * self.theta_bins + ti

    def _polar_density(self, defect_px, die_px, radius, is_noise) -> np.ndarray:
        size = self.r_bins * self.theta_bins
        die_count = np.bincount(self._bin_indices(die_px, radius), minlength=size)
        defect_bins = self._bin_indices(defect_px, radius)
        defect_count = np.bincount(defect_bins, minlength=size)
        noise_count = np.bincount(defect_bins[is_noise], minlength=size)

        polar = np.zeros((POLAR_CHANNELS, size), dtype=np.float32)
        denom = np.maximum(die_count, 1)
        polar[0] = defect_count / denom          # 결함 밀도
        polar[1] = (die_count > 0).astype(np.float32)  # 유효 다이 마스크
        polar[2] = noise_count / denom           # 노이즈 판정 결함 밀도
        return polar.reshape(POLAR_CHANNELS, self.r_bins, self.theta_bins)


## Step 2. 전처리 완료 데이터셋 생성 (build_dataset 함수)

In [ ]:
"""WM811K_defects.pkl -> 학습용 packed .npz 생성 스크립트.

사용 예)
    python src/build_dataset.py \
        --input  /content/drive/MyDrive/ACK2026_Wafer/MIR-WM811K/MIR-WM811K/Python/WM811K_defects.pkl \
        --output /content/drive/MyDrive/ACK2026_Wafer/Train_Model/Training_Data/wafer_v2.npz

가변 길이인 노드/엣지를 하나의 큰 배열 + 오프셋(ptr)으로 묶어 저장한다.
웨이퍼마다 개별 dict 를 피클하던 v1 방식보다 로딩이 빠르고 RAM 을 적게 쓴다.
"""


import argparse
import time

import numpy as np
import pandas as pd



def flatten_label(value):
    """MATLAB 유래의 중첩 배열(['Edge-Ring'] 등)을 문자열로 편다."""
    while isinstance(value, (list, np.ndarray)) and len(value) > 0:
        value = value[0]
    return value




def build_dataset(input_pkl, output_npz, r_bins=64, theta_bins=64, knn_k=8,
                  noise_eps_dies=2.0, noise_min_samples=3, drop_noise=False, limit=0,
                  include_polar=True):
    """원본 pkl 을 읽어 packed .npz 로 저장한다. 노트북에서도 그대로 호출할 수 있다.

    include_polar=False 면 극좌표 이미지를 만들지도 저장하지도 않는다.
    순수 graph 모델만 비교할 때 쓰면 파일이 훨씬 작아지고 전처리도 빨라진다.
    """
    df = pd.read_pickle(input_pkl)
    if limit:
        df = df.iloc[:limit]
    df = df.reset_index(drop=True)  # v1 은 인덱스를 리셋하지 않아 진행률 출력이 깨졌다
    labels_str = df["failureType"].map(flatten_label)
    lot_codes = pd.factorize(df["lotName"].map(flatten_label).astype(str))[0]

    pp = WaferPreprocessorV2(
        r_bins=r_bins, theta_bins=theta_bins, knn_k=knn_k,
        noise_eps_dies=noise_eps_dies, noise_min_samples=noise_min_samples,
        drop_noise=drop_noise, include_polar=include_polar,
    )

    feats, edges, polars, scalars, labels, lots = [], [], [], [], [], []
    skipped = 0
    t0 = time.time()
    for i, wafer_map in enumerate(df["waferMap"]):
        label = labels_str.iloc[i]
        if label not in LABEL_MAP:
            skipped += 1
            continue
        out = pp.process_wafer(wafer_map)
        if out is None:
            skipped += 1
            continue
        feats.append(out["node_feat"])
        edges.append(out["edge_index"])
        if include_polar:
            polars.append(out["polar"])
        scalars.append(out["scalars"])
        labels.append(LABEL_MAP[label])
        lots.append(lot_codes[i])
        if (i + 1) % 2000 == 0:
            print(f"  {i + 1}/{len(df)}  ({time.time() - t0:.0f}s)", flush=True)

    node_ptr = np.concatenate([[0], np.cumsum([len(f) for f in feats])]).astype(np.int64)
    edge_ptr = np.concatenate([[0], np.cumsum([e.shape[1] for e in edges])]).astype(np.int64)

    arrays = dict(
        node_feat=np.concatenate(feats, axis=0),
        node_ptr=node_ptr,
        edge_index=np.concatenate(edges, axis=1) if edges else np.zeros((2, 0), np.int16),
        edge_ptr=edge_ptr,
        scalars=np.stack(scalars),
        labels=np.array(labels, dtype=np.int64),
        lot_ids=np.array(lots, dtype=np.int64),
        theta_bins=theta_bins,
        r_bins=r_bins,
    )
    if include_polar:
        arrays["polar"] = np.stack(polars)
    np.savez_compressed(output_npz, **arrays)

    n = len(labels)
    print(f"\n완료: {n}개 저장, {skipped}개 제외, {time.time() - t0:.0f}s")
    print(f"평균 노드 수 {node_ptr[-1] / n:.1f}, 평균 방향성 엣지 수 {edge_ptr[-1] / n:.1f}")
    print(f"극좌표 이미지: {'포함' if include_polar else '미포함(graph 전용)'}")
    print(f"-> {output_npz}")
    return output_npz


## Step 3. PyG 배치, 회전/반전 증강, 로더 함수 생성

In [ ]:
"""전처리 결과를 PyG 데이터로 공급하는 Dataset + 회전/반전 증강.

[개선 4] 데이터 증강
웨이퍼 결함 라벨은 웨이퍼를 돌리거나 뒤집어도 바뀌지 않는다
(Center/Donut/Edge-Ring/Random/Near-full 은 회전 불변, Loc/Edge-Loc/Scratch 도
반경 위치와 형태가 유지되므로 라벨이 보존된다).
극좌표 표현에서 회전은 theta 축 순환 시프트 한 줄이라 비용이 사실상 0이다.

중요: 그래프 트랙과 이미지 트랙에 **동일한** 변환을 적용해야 한다.
서로 다른 각도로 회전하면 두 트랙이 모순된 정보를 주게 되어 융합이 망가진다.
"""


import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected



class PackedWaferData:
    """`build_dataset.py` 가 만든 .npz 를 메모리에 올린다.

    노드/엣지는 가변 길이라 하나의 큰 배열 + 오프셋(ptr) 형태로 저장한다.
    float16 / int16 로 보관하고 꺼낼 때만 캐스팅해 RAM 사용량을 4분의 1로 줄인다.
    """

    def __init__(self, npz_path: str):
        z = np.load(npz_path, allow_pickle=False)
        self.node_feat = z["node_feat"]      # [총 노드 수, F] float16
        self.node_ptr = z["node_ptr"]        # [N+1]
        self.edge_index = z["edge_index"]    # [2, 총 엣지 수] int16 (그래프 로컬 인덱스)
        self.edge_ptr = z["edge_ptr"]        # [N+1]
        # graph 전용 npz(build_dataset(..., include_polar=False))에는 polar 가 없다.
        self.polar = z["polar"] if "polar" in z.files else None
        self.scalars = z["scalars"]          # [N, S] float32
        self.labels = z["labels"]            # [N] int64
        self.lot_ids = z["lot_ids"]          # [N] int64 (lot 단위 분할용)
        self.theta_bins = int(z["theta_bins"])

    def __len__(self):
        return len(self.labels)

    def get(self, i: int):
        ns, ne = self.node_ptr[i], self.node_ptr[i + 1]
        es, ee = self.edge_ptr[i], self.edge_ptr[i + 1]
        polar = (torch.from_numpy(self.polar[i].astype(np.float32))
                 if self.polar is not None else None)
        return (
            torch.from_numpy(self.node_feat[ns:ne].astype(np.float32)),
            torch.from_numpy(self.edge_index[:, es:ee].astype(np.int64)),
            polar,
            torch.from_numpy(self.scalars[i]),
            int(self.labels[i]),
        )


class WaferDualTrackDataset(torch.utils.data.Dataset):
    """PyG `Data` 를 돌려주는 Dataset. `augment=True` 일 때만 증강한다.

    Parameters
    ----------
    augment : 학습 세트에만 True. 검증/테스트에는 반드시 False.
    rotate : theta 빈 단위 무작위 회전.
    flip : 무작위 좌우 반전(거울상).
    """

    def __init__(self, packed: PackedWaferData, indices, augment=False,
                 rotate=True, flip=True, seed=0):
        self.packed = packed
        self.indices = np.asarray(indices)
        self.augment = augment
        self.rotate = rotate
        self.flip = flip
        self.theta_bins = packed.theta_bins
        self._rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, edge_index, polar, scalars, y = self.packed.get(self.indices[i])

        if self.augment:
            if self.flip and self._rng.random() < 0.5:
                x, polar = _apply_flip(x, polar)
            if self.rotate:
                shift = int(self._rng.integers(0, self.theta_bins))
                if shift:
                    x, polar = _apply_rotation(x, polar, shift, self.theta_bins)

        # 저장은 방향성 k-NN 엣지로 하고, 여기서 대칭화한다(중복 제거 포함).
        if edge_index.numel():
            edge_index = to_undirected(edge_index, num_nodes=x.size(0))

        data = Data(
            x=x,
            edge_index=edge_index,
            scalars=scalars.unsqueeze(0),      # [1, S] -> [B, S]
            y=torch.tensor([y], dtype=torch.long),
        )
        if polar is not None:
            data.polar = polar.unsqueeze(0)    # [1, C, R, T] -> 배치 시 [B, C, R, T]
        return data


# ----------------------------------------------------------------------
# 증강 연산: 두 트랙이 정확히 같은 변환을 받도록 한 곳에서 정의한다.
# ----------------------------------------------------------------------
def _apply_rotation(x: torch.Tensor, polar: torch.Tensor | None, shift: int, theta_bins: int):
    """theta 축으로 `shift` 빈 회전.

    극좌표 인덱스가 ti = ((theta + pi) / 2pi * T) % T 이므로 점을
    +phi 만큼 돌리면 ti -> ti + shift 가 된다. 이는 torch.roll 과 정확히 일치한다.

    polar 가 None(graph 전용 데이터)이면 그래프 쪽만 회전한다.
    """
    phi = 2.0 * np.pi * shift / theta_bins
    c, s = np.cos(phi), np.sin(phi)
    x = x.clone()
    px, py = x[:, FEAT_X].clone(), x[:, FEAT_Y].clone()
    x[:, FEAT_X] = c * px - s * py
    x[:, FEAT_Y] = s * px + c * py
    cx, sy = x[:, FEAT_COS].clone(), x[:, FEAT_SIN].clone()
    x[:, FEAT_COS] = c * cx - s * sy
    x[:, FEAT_SIN] = s * cx + c * sy
    if polar is None:
        return x, None
    return x, torch.roll(polar, shifts=shift, dims=-1)


def _apply_flip(x: torch.Tensor, polar: torch.Tensor | None):
    """y -> -y 거울상. 극좌표에서는 theta -> -theta.

    u = (theta + pi) / 2pi * T 이고 비닝이 round(u) 이므로 반전 후에는
    round(T - u) = T - round(u), 즉 ti -> (-ti) % T 가 된다.
    torch.flip 은 i -> T-1-i 이라 한 칸 모자라므로 roll(+1) 로 보정한다.
    """
    x = x.clone()
    x[:, FEAT_Y] = -x[:, FEAT_Y]
    x[:, FEAT_SIN] = -x[:, FEAT_SIN]
    if polar is None:
        return x, None
    return x, torch.roll(torch.flip(polar, dims=[-1]), shifts=1, dims=-1)


# ----------------------------------------------------------------------
# 학습 코드에 바로 물릴 수 있는 로더 구성 헬퍼.
# 모델은 포함하지 않는다 -- 어떤 graph 모델이든 이 로더를 그대로 받아 쓰면 된다.
# ----------------------------------------------------------------------
def make_splits(labels, lot_ids, mode="stratified", holdout_test=False, seed=42):
    """(train, val, test) 인덱스를 만든다.

    mode="stratified" : v1 과 동일한 무작위 층화 분할. 같은 lot 이 양쪽에 섞인다.
    mode="lot"        : 같은 lot 은 train/val/test 중 한 곳에만. 누수가 사라진다.
    holdout_test=False: test = val (v1 호환). True 면 70/15/15 3분할.
    """
    from sklearn.model_selection import GroupShuffleSplit, train_test_split

    idx = np.arange(len(labels))

    def split(indices, frac):
        if mode == "lot":
            gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
            a, b = next(gss.split(indices, labels[indices], lot_ids[indices]))
        else:
            a, b = train_test_split(np.arange(len(indices)), test_size=frac,
                                    stratify=labels[indices], random_state=seed)
        return indices[a], indices[b]

    if not holdout_test:
        train_idx, val_idx = split(idx, 0.2)
        return train_idx, val_idx, val_idx
    train_idx, rest = split(idx, 0.3)
    val_idx, test_idx = split(rest, 0.5)
    return train_idx, val_idx, test_idx


def _worker_init(worker_id):
    """DataLoader 워커마다 증강 RNG 를 다시 뿌린다(안 하면 모든 워커가 같은 난수를 쓴다)."""
    info = torch.utils.data.get_worker_info()
    info.dataset._rng = np.random.default_rng(torch.initial_seed() % (2**32))


def make_loaders(npz_path, batch_size=64, augment=True, split="stratified",
                 holdout_test=False, seed=42, num_workers=2):
    """npz 하나로 (train_loader, val_loader, test_loader, packed) 를 만든다.

    증강은 train 에만 걸린다. val/test 에 증강이 들어가면 채점 문제 자체가 바뀐다.

    사용 예)
        tr, va, te, packed = make_loaders("wafer_graph.npz")
        model = MyGraphNet(in_dim=NUM_NODE_FEATURES, num_classes=8)
        for batch in tr:
            out = model(batch.x, batch.edge_index, batch.batch)
    """
    from torch_geometric.loader import DataLoader

    packed = PackedWaferData(npz_path)
    train_idx, val_idx, test_idx = make_splits(
        packed.labels, packed.lot_ids, split, holdout_test, seed)

    train_set = WaferDualTrackDataset(packed, train_idx, augment=augment, seed=seed)
    val_set = WaferDualTrackDataset(packed, val_idx, augment=False)
    test_set = WaferDualTrackDataset(packed, test_idx, augment=False)

    common = dict(num_workers=num_workers, persistent_workers=num_workers > 0)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              worker_init_fn=_worker_init if num_workers else None, **common)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, **common)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, **common)
    print(f"train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}"
          f"  (분할: {split}, 증강: {augment}, 극좌표: {packed.polar is not None})")
    return train_loader, val_loader, test_loader, packed


def class_weights(labels, num_classes=8):
    """sklearn 의 'balanced' 와 같은 식. 클래스가 빠져도 길이를 항상 보장한다."""
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    return np.where(counts > 0, len(labels) / (num_classes * np.maximum(counts, 1)), 0.0)


## Step 4. 증강 테스트

In [ ]:
"""증강 정합성 검증.

회전/반전은 그래프 트랙과 이미지 트랙에 **똑같이** 적용돼야 한다. 어긋나면 두 트랙이
모순된 정보를 주게 되어 융합 성능이 오히려 떨어진다. 여기서 검증하는 것:

1. 회전 후 노드 좌표로 다시 계산한 theta 빈 == torch.roll 로 이동한 빈
2. 반전 후에도 동일
3. 회전/반전이 반경 r 과 노드 개수를 보존
4. cos/sin 특징이 회전 후에도 (x, y) 와 일치

실행: python src/test_augmentation.py <WM811K_defects.pkl>
"""


import sys

import numpy as np
import torch


T = 64


def theta_bin(x, y):
    """wafer_preprocessing._bin_indices 와 동일한 theta 비닝."""
    theta = np.arctan2(y, x)
    return np.floor((theta + np.pi) / (2 * np.pi) * T + 0.5).astype(np.int64) % T


def bin_gap(a, b):
    """두 theta 빈 사이의 순환 거리(칸 수)."""
    d = np.abs(a - b) % T
    return np.minimum(d, T - d)


def check(wafer_map, pp):
    out = pp.process_wafer(wafer_map)
    if out is None:
        return None
    x = torch.from_numpy(out["node_feat"].astype(np.float32))
    polar = torch.from_numpy(out["polar"].astype(np.float32))
    base_bin = theta_bin(x[:, FEAT_X].numpy(), x[:, FEAT_Y].numpy())
    # 웨이퍼 정중앙(r = 0)의 노드는 각도가 정의되지 않아 회전에 불변이다. 비교에서 제외.
    has_angle = x[:, FEAT_R].numpy() > 1e-3
    errs = []

    for shift in (1, 7, 16, 33, 63):
        xr, pr = _apply_rotation(x, polar, shift, T)
        got = theta_bin(xr[:, FEAT_X].numpy(), xr[:, FEAT_Y].numpy())
        want = (base_bin + shift) % T
        # node_feat 이 float16 이라 빈 경계에서 ±1 칸까지는 허용한다.
        # 2칸 이상 벌어지면 변환 자체가 틀린 것이다.
        gap = bin_gap(got, want)[has_angle]
        if gap.max() > 1:
            errs.append(f"rot{shift}: theta 빈 최대 {gap.max()}칸 어긋남")
        elif (gap > 0).mean() > 0.02:
            errs.append(f"rot{shift}: 경계 오차 과다 {(gap > 0).mean():.3f}")
        if not torch.allclose(pr, torch.roll(polar, shift, dims=-1)):
            errs.append(f"rot{shift}: polar roll 불일치")
        if not torch.allclose(xr[:, FEAT_R], x[:, FEAT_R], atol=1e-5):
            errs.append(f"rot{shift}: 반경이 변함")
        # cos/sin 이 회전 후 좌표와 여전히 일치하는가
        r = xr[:, FEAT_R].clamp(min=1e-6)
        if not torch.allclose(xr[:, FEAT_COS] * r, xr[:, FEAT_X], atol=1e-3):
            errs.append(f"rot{shift}: cos 특징 불일치")
        if not torch.allclose(xr[:, FEAT_SIN] * r, xr[:, FEAT_Y], atol=1e-3):
            errs.append(f"rot{shift}: sin 특징 불일치")

    xf, pf = _apply_flip(x, polar)
    got = theta_bin(xf[:, FEAT_X].numpy(), xf[:, FEAT_Y].numpy())
    want = (-base_bin) % T              # round 비닝에서는 ti -> (-ti) % T
    gap = bin_gap(got, want)[has_angle]
    if gap.max() > 1:
        errs.append(f"flip: theta 빈 최대 {gap.max()}칸 어긋남")
    elif (gap > 0).mean() > 0.02:
        errs.append(f"flip: 경계 오차 과다 {(gap > 0).mean():.3f}")
    if not torch.allclose(pf, torch.roll(torch.flip(polar, dims=[-1]), 1, dims=-1)):
        errs.append("flip: polar 불일치")
    if not torch.allclose(xf[:, FEAT_R], x[:, FEAT_R], atol=1e-5):
        errs.append("flip: 반경이 변함")
    # 반전은 결함 밀도의 총합을 보존해야 한다
    if not torch.allclose(pf.sum(), polar.sum(), atol=1e-3):
        errs.append("flip: 밀도 총합 불일치")
    return errs


def run_augmentation_test(pkl_path, n_samples=200, seed=0):
    """웨이퍼 n장을 뽑아 증강 정합성을 검사한다. 실패 건수를 돌려준다."""
    import pandas as pd

    df = pd.read_pickle(pkl_path)
    pp = WaferPreprocessorV2(theta_bins=T)
    rng = np.random.default_rng(seed)
    sample = rng.choice(len(df), size=min(n_samples, len(df)), replace=False)

    failures = 0
    for i in sample:
        errs = check(df.iloc[i]["waferMap"], pp)
        if errs:
            failures += 1
            print(f"[웨이퍼 {i}] " + "; ".join(errs))
    print(f"\n{len(sample)}개 중 {failures}개 실패")
    return failures
